# Turbine defect detector

Runs end to end. The only thing you provide is a Roboflow API key.

### Setup, once
1. **Add-ons > Secrets** > add `ROBOFLOW_API_KEY` (from app.roboflow.com/settings/api) and
   **tick it for this notebook**. Kaggle secrets are per notebook, not per account.
2. Optionally add a GitHub token the same way, so the finished model commits itself.
3. **Settings > Accelerator: GPU T4 x2**, **Internet: On**.
4. **Save Version > Save & Run All (Commit)**. Never a Draft Session: a draft dies with the
   browser and takes the weights with it.

### What it does
Discovers every project in your Roboflow workspace, downloads the turbine ones, merges them
into one dataset while deduplicating photographs that appear in more than one source, audits
the result against eight structural checks, trains, measures, and exports.

It stops before training if the audit fails. That is the point of it: the previous dataset
scored mAP50 0.759 and reported no defects on a turbine with a blade snapped in half, and no
metric computed inside its own distribution could have caught that.

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime

In [ ]:
# Set this True to inspect the dataset without spending GPU time on it.
#
# Everything up to and including the audit runs on CPU in about ten minutes, and answers the
# questions that decide whether training is worth starting: how many distinct photographs are
# really here, whether the sources overlap, whether any class is too thin to learn, and
# whether in-domain negatives exist at all. Training on a dataset before knowing those is how
# the previous model happened.
STOP_AFTER_AUDIT = True

import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)

HOWTO = ("\n  Settings > Accelerator > GPU T4 x2, accept the restart, then Run All."
         "\n  Not P100: torch dropped Pascal support, so it is detected but unusable.")

if not torch.cuda.is_available():
    raise SystemExit("\n" + "=" * 74
                     + "\nNO GPU. Training would take days on CPU.\n"
                     + "=" * 74 + HOWTO)

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
    raise SystemExit(f"\n{name} is sm_{major}{minor}, which this torch cannot use." + HOWTO)

print(f"cuda: {name} (sm_{major}{minor})")
print(f"vram: {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB")
print("\nGPU OK.")


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working/drone-inspection")
if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/abyyworld/Drone-visualisation-training.git",
                    str(WORK)], check=True)
os.chdir(WORK); sys.path.insert(0, str(WORK))
assert (WORK / "tools" / "merge_datasets.py").exists()
print("repo ->", WORK)

## 1. Get the data

In [ ]:
# Download the four Roboflow projects.
#
# Workspace and project ids are explicit so a rename in the Roboflow UI produces a clear
# failure rather than a silently smaller dataset. If any is missing, the workspace is listed
# so the id can be corrected.
import re
from pathlib import Path

WORKSPACE = "akbars-workspace-hcecg"
PROJECTS = [
    "wind-turbine-defect-detection-qinzg",
    "crackk-xufde-p7gaj",
    "damage-nuvjn-qmfec",
    "wind-turbine-blades-6oc4y",
]
RAW = Path("/kaggle/working/raw")

from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

# Kaggle offers no way to list secret labels, so a mistyped one is indistinguishable from
# none at all. Try the names people actually use rather than making one string load-bearing.
SECRET_NAMES = ("ROBOFLOW_API_KEY", "ROBOFLOW_KEY", "roboflow key", "roboflow_key",
                "roboflow", "ROBOFLOW")
client, key = UserSecretsClient(), None
for label in SECRET_NAMES:
    try:
        key = (client.get_secret(label) or "").strip()
        if key:
            print(f"Using the key from Kaggle secret {label!r}\n")
            break
    except Exception:
        continue
assert key, ("No Roboflow key found. Tried: " + ", ".join(SECRET_NAMES)
             + "\nAdd-ons > Secrets > add it, and tick it for this notebook.")

# A 401 here means the value is wrong, not that the secret is missing. Roboflow issues two
# keys and only one of them works with the SDK: the Private API Key, not the Publishable Key.
try:
    rf = Roboflow(api_key=key)
except RuntimeError as error:
    raise SystemExit(
        f"Roboflow rejected the key ({len(key)} characters).\n\n"
        "Get the PRIVATE API KEY, not the publishable one:\n"
        "  app.roboflow.com > Settings > API Keys > Private API Key\n\n"
        "Then in Kaggle: Add-ons > Secrets > edit ROBOFLOW_API_KEY > Save, and make sure it\n"
        "is ticked for this notebook. Paste with no leading or trailing spaces; the value is\n"
        "stripped here but a newline pasted into the middle would still break it.\n\n"
        f"Roboflow said: {error}"
    ) from None

workspace = rf.workspace(WORKSPACE)
RAW.mkdir(parents=True, exist_ok=True)

downloaded, missing = [], []
for pid in PROJECTS:
    target = RAW / pid
    if (target / "data.yaml").exists():
        downloaded.append(target)
        print(f"  {pid}: already present")
        continue
    try:
        project = workspace.project(pid)
        # Version numbering is not always 1..n, so ask rather than assume; take the newest.
        numbers = []
        for v in project.versions():
            raw = str(getattr(v, "version", None) or getattr(v, "id", ""))
            digits = re.search(r"(\d+)$", raw)
            if digits:
                numbers.append(int(digits.group(1)))
        assert numbers, "no versions - create one in Roboflow first"
        print(f"  {pid}: downloading version {max(numbers)} ...")
        project.version(max(numbers)).download("yolov11", location=str(target))
        downloaded.append(target)
    except Exception as error:
        print(f"  {pid}: FAILED - {error}")
        missing.append(pid)

if missing:
    print("\nProjects visible in this workspace:")
    for p in workspace.projects():
        print(f"    {p}")

assert downloaded, "nothing downloaded"
print(f"\n{len(downloaded)} of {len(PROJECTS)} datasets in {RAW}\n")
for d in downloaded:
    counts = {s: len(list((d / s / "images").glob("*"))) if (d / s / "images").is_dir() else 0
              for s in ("train", "valid", "test")}
    print(f"    {d.name:<40}{sum(counts.values()):>6} images  {counts}")

SOURCES = [str(d) for d in downloaded]


## 2. Merge

Four exports of one subject are not four datasets. They overlap, they disagree on names, and
each is separately inflated by its exporter's augmentation. This deduplicates by image
content, harmonises the class names, and splits by scene so two versions of one photograph
cannot land on opposite sides.

The grouping is proposed, not decided: any class name it does not recognise stops the merge
rather than being guessed at.

In [ ]:
import json

# Discovery pass: report what every source contains and propose a grouping.
subprocess.run(["python3", "tools/merge_datasets.py"]
               + sum([["--source", s] for s in SOURCES], []), check=True)

MAP = Path("docs/class-map.json")
spec = json.loads(MAP.read_text())
unplaced = [k for k, v in spec["classes"].items() if v == "PLACE_ME"]

assert not unplaced, (
    "These class names were not recognised, so merging would guess at them:\n  "
    + "\n  ".join(unplaced)
    + "\n\nDecide what each one is, edit docs/class-map.json in the repo, commit, re-run."
)

print("\nEvery class placed. Merging.\n")
subprocess.run(["python3", "tools/merge_datasets.py"]
               + sum([["--source", s] for s in SOURCES], [])
               + ["--mapping", str(MAP), "--out", "/kaggle/working/turbine", "--copy"],
               check=True)


## 3. Audit

Eight checks. Training on a dataset that fails one is how the last model happened.

In [ ]:
result = subprocess.run(["python3", "tools/audit_dataset.py", "/kaggle/working/turbine"])
assert result.returncode == 0, (
    "The audit failed. Do not train on this. Read the failing checks above: each one names a "
    "structural problem that no metric computed afterwards would reveal."
)

## 4. Train

In [ ]:
if STOP_AFTER_AUDIT:
    raise SystemExit(
        "Stopping before training, as configured.\n\n"
        "Read the three numbers above before deciding:\n"
        "  1. files -> distinct scenes. A ratio above about 2.5 means the sources are mostly\n"
        "     augmented copies, and the real dataset is the second number.\n"
        "  2. per class instance counts. Under roughly 300 in train, a class is sampling\n"
        "     noise; it is better dropped than shipped, because a class that looks like it\n"
        "     works and does not is worse than one that is absent.\n"
        "  3. images with boxes versus images without. The difference is your in-domain\n"
        "     negatives, and they are what teach the model that a healthy blade is healthy.\n"
        "     If that difference is near zero, expect false positives on clean blades, which\n"
        "     is the failure the previous model had.\n\n"
        "Set STOP_AFTER_AUDIT = False and re-run when the answers look right."
    )

subprocess.run(["python3", "tools/train_turbine.py",
                "--data", "/kaggle/working/turbine/data.yaml",
                "--project", "/kaggle/working/runs", "--name", "turbine"], check=True)


## 5. Measure

Per class, not aggregate. Then the check that matters more: how often the model draws a box
on a healthy blade, where every box is by construction a false positive.

In [ ]:
subprocess.run(["python3", "tools/evaluate.py",
                "/kaggle/working/runs/turbine/weights/best.pt",
                "--data", "/kaggle/working/turbine/data.yaml", "--split", "test",
                "--imgsz", "960", "--out", "/kaggle/working/runs/eval"], check=True)

## 6. Export

In [ ]:
subprocess.run(["python3", "tools/export_variants.py",
                "/kaggle/working/runs/turbine/weights/best.pt",
                "--data", "/kaggle/working/turbine/data.yaml",
                "--name", "turbine", "--imgsz", "960"], check=True)

subprocess.run(["python3", "tools/false_positive_check.py",
                "--model", "web/models/turbine.onnx", "--limit", "300"], check=False)

## 7. Publish

Commits the model and its metrics to the working branch if a GitHub token is set. Pushes to
the branch, never to main, so the live site is unchanged until you merge.

In [ ]:
subprocess.run(["python3", "tools/publish_results.py", "--name", "turbine",
                "--metrics", "/kaggle/working/runs/eval"], check=False)